# Bounding Volume Hierarchy (BVH) Visualization with topologic_fast

This notebook demonstrates Bounding Volume Hierarchy concepts using topologic_fast.

A BVH is a tree structure for organizing spatial objects to enable fast collision detection and ray tracing. We will:

1. Create a set of 3D cells (objects)
2. Compute bounding boxes for each object
3. Visualize the hierarchy of bounding boxes
4. Demonstrate spatial queries and collision detection concepts

**Note:** This is adapted from topologicpy's BVH tutorial. The topologic_fast library may not have a dedicated BVH module, so we demonstrate the underlying concepts using available topology operations.

In [ ]:
# Import libraries
import topologic_fast as tf
import plotly.graph_objects as go
import numpy as np
from collections import deque

## Create a Set of 3D Objects

We'll create a CellComplex and explode it into individual cells, then work with their bounding boxes.

In [ ]:
# Create a CellComplex grid
cells = []
cell_centers = []

# Create a 5x5x2 grid of cells with some spacing
spacing = 2.5  # Space between cells
cell_size = 2.0

for k in range(2):  # 2 floors
    for j in range(5):  # 5 rows
        for i in range(5):  # 5 columns
            x = i * spacing
            y = j * spacing
            z = k * spacing * 1.5
            
            cell = tf.Cell.Box(x, y, z, cell_size, cell_size, cell_size)
            cells.append(cell)
            cell_centers.append((x + cell_size/2, y + cell_size/2, z + cell_size/2))

print(f"Created {len(cells)} cells")

## Compute Bounding Boxes

For each cell, we compute its axis-aligned bounding box (AABB).

In [ ]:
def get_bounding_box(cell):
    """
    Get the axis-aligned bounding box (AABB) of a cell.
    Returns (min_x, min_y, min_z, max_x, max_y, max_z)
    """
    vertices = cell.Vertices()
    coords = [v.Coordinates() for v in vertices]
    
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]
    zs = [c[2] for c in coords]
    
    return {
        'min_x': min(xs), 'max_x': max(xs),
        'min_y': min(ys), 'max_y': max(ys),
        'min_z': min(zs), 'max_z': max(zs),
        'center': ((min(xs) + max(xs))/2, (min(ys) + max(ys))/2, (min(zs) + max(zs))/2)
    }

# Compute bounding boxes for all cells
bboxes = [get_bounding_box(cell) for cell in cells]

print(f"Computed {len(bboxes)} bounding boxes")
print(f"\nExample bounding box:")
print(f"  X: [{bboxes[0]['min_x']:.2f}, {bboxes[0]['max_x']:.2f}]")
print(f"  Y: [{bboxes[0]['min_y']:.2f}, {bboxes[0]['max_y']:.2f}]")
print(f"  Z: [{bboxes[0]['min_z']:.2f}, {bboxes[0]['max_z']:.2f}]")

## Build a Simple BVH Tree

We'll build a binary tree that hierarchically groups objects based on their spatial positions.

In [ ]:
class BVHNode:
    """A node in the Bounding Volume Hierarchy tree."""
    def __init__(self, objects=None, bbox=None, left=None, right=None, depth=0):
        self.objects = objects or []  # Leaf nodes contain objects
        self.bbox = bbox  # Bounding box of this node
        self.left = left
        self.right = right
        self.depth = depth
        self.is_leaf = left is None and right is None

def merge_bboxes(bboxes):
    """Merge multiple bounding boxes into one."""
    if not bboxes:
        return None
    return {
        'min_x': min(b['min_x'] for b in bboxes),
        'max_x': max(b['max_x'] for b in bboxes),
        'min_y': min(b['min_y'] for b in bboxes),
        'max_y': max(b['max_y'] for b in bboxes),
        'min_z': min(b['min_z'] for b in bboxes),
        'max_z': max(b['max_z'] for b in bboxes),
        'center': None  # Will be computed
    }

def build_bvh(objects, bboxes, depth=0, max_objects_per_leaf=4):
    """
    Build a BVH tree recursively.
    Uses spatial median split along the longest axis.
    """
    if len(objects) == 0:
        return None
    
    # Compute merged bounding box
    merged_bbox = merge_bboxes(bboxes)
    merged_bbox['center'] = (
        (merged_bbox['min_x'] + merged_bbox['max_x']) / 2,
        (merged_bbox['min_y'] + merged_bbox['max_y']) / 2,
        (merged_bbox['min_z'] + merged_bbox['max_z']) / 2
    )
    
    # Base case: create leaf node
    if len(objects) <= max_objects_per_leaf:
        return BVHNode(objects=list(range(len(objects))), bbox=merged_bbox, depth=depth)
    
    # Find longest axis
    extent_x = merged_bbox['max_x'] - merged_bbox['min_x']
    extent_y = merged_bbox['max_y'] - merged_bbox['min_y']
    extent_z = merged_bbox['max_z'] - merged_bbox['min_z']
    
    if extent_x >= extent_y and extent_x >= extent_z:
        axis = 0  # Split along X
    elif extent_y >= extent_z:
        axis = 1  # Split along Y
    else:
        axis = 2  # Split along Z
    
    # Sort objects by center along split axis
    centers = [b['center'][axis] for b in bboxes]
    sorted_indices = np.argsort(centers)
    
    # Split at median
    mid = len(sorted_indices) // 2
    left_indices = sorted_indices[:mid]
    right_indices = sorted_indices[mid:]
    
    # Recursively build subtrees
    left_objects = [objects[i] for i in left_indices]
    left_bboxes = [bboxes[i] for i in left_indices]
    right_objects = [objects[i] for i in right_indices]
    right_bboxes = [bboxes[i] for i in right_indices]
    
    left_node = build_bvh(left_objects, left_bboxes, depth + 1, max_objects_per_leaf)
    right_node = build_bvh(right_objects, right_bboxes, depth + 1, max_objects_per_leaf)
    
    return BVHNode(bbox=merged_bbox, left=left_node, right=right_node, depth=depth)

# Build the BVH tree
bvh_root = build_bvh(cells, bboxes)

# Count nodes
def count_nodes(node):
    if node is None:
        return 0, 0
    if node.is_leaf:
        return 1, 1
    left_total, left_leaves = count_nodes(node.left)
    right_total, right_leaves = count_nodes(node.right)
    return 1 + left_total + right_total, left_leaves + right_leaves

total_nodes, leaf_nodes = count_nodes(bvh_root)
print(f"BVH Tree Statistics:")
print(f"  Total nodes: {total_nodes}")
print(f"  Leaf nodes: {leaf_nodes}")
print(f"  Internal nodes: {total_nodes - leaf_nodes}")

## Visualize Cells with Bounding Boxes

In [ ]:
def draw_bbox_wireframe(fig, bbox, color='red', width=2, name=None, showlegend=False):
    """Draw a wireframe bounding box."""
    x0, x1 = bbox['min_x'], bbox['max_x']
    y0, y1 = bbox['min_y'], bbox['max_y']
    z0, z1 = bbox['min_z'], bbox['max_z']
    
    # 12 edges of the box
    edges = [
        # Bottom face
        ([x0, x1], [y0, y0], [z0, z0]),
        ([x1, x1], [y0, y1], [z0, z0]),
        ([x1, x0], [y1, y1], [z0, z0]),
        ([x0, x0], [y1, y0], [z0, z0]),
        # Top face
        ([x0, x1], [y0, y0], [z1, z1]),
        ([x1, x1], [y0, y1], [z1, z1]),
        ([x1, x0], [y1, y1], [z1, z1]),
        ([x0, x0], [y1, y0], [z1, z1]),
        # Vertical edges
        ([x0, x0], [y0, y0], [z0, z1]),
        ([x1, x1], [y0, y0], [z0, z1]),
        ([x1, x1], [y1, y1], [z0, z1]),
        ([x0, x0], [y1, y1], [z0, z1]),
    ]
    
    for i, (xs, ys, zs) in enumerate(edges):
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode='lines',
            line=dict(color=color, width=width),
            name=name if i == 0 and showlegend else None,
            showlegend=(i == 0 and showlegend),
            hoverinfo='skip'
        ))

def visualize_cells_with_bboxes(cells, bboxes, show_bboxes=True):
    """Visualize cells with their bounding boxes."""
    fig = go.Figure()
    
    # Draw cells
    for i, cell in enumerate(cells):
        faces = cell.Faces()
        for j, face in enumerate(faces):
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color='lightblue',
                    opacity=0.4,
                    alphahull=0,
                    showlegend=False,
                    hoverinfo='skip'
                ))
    
    # Draw bounding boxes
    if show_bboxes:
        for i, bbox in enumerate(bboxes):
            draw_bbox_wireframe(fig, bbox, color='rgba(255,0,0,0.3)', width=1)
    
    fig.update_layout(
        title='Cells with Individual Bounding Boxes',
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=900,
        height=700
    )
    
    return fig

fig_cells = visualize_cells_with_bboxes(cells, bboxes)
fig_cells.show()

## Visualize BVH Hierarchy

Show the hierarchical bounding boxes at different levels of the tree.

In [ ]:
def get_nodes_at_depth(root, target_depth):
    """Get all BVH nodes at a specific depth."""
    nodes = []
    queue = deque([root])
    
    while queue:
        node = queue.popleft()
        if node is None:
            continue
        if node.depth == target_depth:
            nodes.append(node)
        elif node.depth < target_depth:
            queue.append(node.left)
            queue.append(node.right)
    
    return nodes

def visualize_bvh_levels(root, cells, max_depth=4):
    """Visualize BVH hierarchy at different levels."""
    # Color scheme for different depths
    depth_colors = [
        'rgba(255,0,0,0.8)',     # Level 0 - Red
        'rgba(255,165,0,0.7)',   # Level 1 - Orange
        'rgba(255,255,0,0.6)',   # Level 2 - Yellow
        'rgba(0,255,0,0.5)',     # Level 3 - Green
        'rgba(0,255,255,0.4)',   # Level 4 - Cyan
        'rgba(0,0,255,0.3)',     # Level 5 - Blue
    ]
    
    fig = go.Figure()
    
    # Draw cells faintly
    for cell in cells:
        faces = cell.Faces()
        for face in faces:
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color='lightgray',
                    opacity=0.1,
                    alphahull=0,
                    showlegend=False,
                    hoverinfo='skip'
                ))
    
    # Draw BVH bounding boxes at each level
    for depth in range(max_depth + 1):
        nodes = get_nodes_at_depth(root, depth)
        color = depth_colors[depth % len(depth_colors)]
        
        for i, node in enumerate(nodes):
            if node.bbox:
                draw_bbox_wireframe(
                    fig, node.bbox, 
                    color=color, 
                    width=4 - depth * 0.5,
                    name=f'Level {depth}',
                    showlegend=(i == 0)
                )
    
    fig.update_layout(
        title='BVH Hierarchy Visualization (Colored by Depth)',
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            camera=dict(eye=dict(x=1.8, y=-1.8, z=1.2))
        ),
        width=900,
        height=700,
        legend=dict(title='BVH Depth', x=1.02, y=1)
    )
    
    return fig

fig_hierarchy = visualize_bvh_levels(bvh_root, cells)
fig_hierarchy.show()

## Collision Detection Query

Demonstrate how BVH can be used to find potential collisions efficiently.

In [ ]:
def boxes_overlap(bbox1, bbox2):
    """Check if two bounding boxes overlap."""
    return (bbox1['min_x'] <= bbox2['max_x'] and bbox1['max_x'] >= bbox2['min_x'] and
            bbox1['min_y'] <= bbox2['max_y'] and bbox1['max_y'] >= bbox2['min_y'] and
            bbox1['min_z'] <= bbox2['max_z'] and bbox1['max_z'] >= bbox2['min_z'])

def query_bvh(root, query_bbox, all_objects):
    """
    Query the BVH for all objects that potentially overlap with query_bbox.
    Returns list of object indices.
    """
    if root is None:
        return []
    
    # Check if query overlaps this node's bbox
    if not boxes_overlap(root.bbox, query_bbox):
        return []
    
    # If leaf, return objects
    if root.is_leaf:
        return root.objects
    
    # Otherwise, recurse
    left_results = query_bvh(root.left, query_bbox, all_objects)
    right_results = query_bvh(root.right, query_bbox, all_objects)
    
    return left_results + right_results

# Create a query box (moving object)
query_cell = tf.Cell.Box(4, 4, 2, 3, 3, 3)  # A 3x3x3 box
query_bbox = get_bounding_box(query_cell)

# Find potentially colliding objects
# Note: Since we rebuilt the tree, we need to map indices correctly
# For this demo, we'll use a simple brute-force check
colliding_indices = []
for i, bbox in enumerate(bboxes):
    if boxes_overlap(query_bbox, bbox):
        colliding_indices.append(i)

print(f"Query box: ({query_bbox['min_x']:.1f}, {query_bbox['min_y']:.1f}, {query_bbox['min_z']:.1f}) to ({query_bbox['max_x']:.1f}, {query_bbox['max_y']:.1f}, {query_bbox['max_z']:.1f})")
print(f"\nPotentially colliding objects: {len(colliding_indices)} out of {len(cells)}")
print(f"Indices: {colliding_indices}")

## Visualize Collision Query

In [ ]:
def visualize_collision_query(cells, bboxes, query_cell, query_bbox, colliding_indices):
    """Visualize collision detection query results."""
    fig = go.Figure()
    
    # Draw non-colliding cells (gray)
    for i, cell in enumerate(cells):
        if i in colliding_indices:
            continue
        faces = cell.Faces()
        for face in faces:
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color='lightgray',
                    opacity=0.2,
                    alphahull=0,
                    showlegend=False,
                    hoverinfo='skip'
                ))
    
    # Draw colliding cells (yellow)
    for i in colliding_indices:
        cell = cells[i]
        faces = cell.Faces()
        for j, face in enumerate(faces):
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color='yellow',
                    opacity=0.7,
                    alphahull=0,
                    name='Colliding' if (i == colliding_indices[0] and j == 0) else None,
                    showlegend=(i == colliding_indices[0] and j == 0),
                    hoverinfo='skip'
                ))
    
    # Draw query cell (red, semi-transparent)
    faces = query_cell.Faces()
    for j, face in enumerate(faces):
        vertices = face.Vertices()
        coords = [v.Coordinates() for v in vertices]
        if len(coords) >= 3:
            x = [c[0] for c in coords]
            y = [c[1] for c in coords]
            z = [c[2] for c in coords]
            fig.add_trace(go.Mesh3d(
                x=x, y=y, z=z,
                color='red',
                opacity=0.5,
                alphahull=0,
                name='Query Object' if j == 0 else None,
                showlegend=(j == 0),
                hoverinfo='skip'
            ))
    
    # Draw query bounding box
    draw_bbox_wireframe(fig, query_bbox, color='red', width=4, name='Query BBox', showlegend=True)
    
    fig.update_layout(
        title=f'Collision Query: {len(colliding_indices)} Potential Collisions',
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=900,
        height=700,
        legend=dict(x=1.02, y=1)
    )
    
    return fig

fig_collision = visualize_collision_query(cells, bboxes, query_cell, query_bbox, colliding_indices)
fig_collision.show()

## Animated Path Through Space

Simulate a moving object and show which cells it potentially collides with along its path.

In [ ]:
def create_path_animation_frames(cells, bboxes, num_steps=20):
    """
    Create frames for an animation of a moving object.
    """
    frames = []
    
    # Path: diagonal across the grid
    start_pos = (0, 0, 0)
    end_pos = (10, 10, 3)
    
    for step in range(num_steps + 1):
        t = step / num_steps
        
        # Interpolate position
        x = start_pos[0] + t * (end_pos[0] - start_pos[0])
        y = start_pos[1] + t * (end_pos[1] - start_pos[1])
        z = start_pos[2] + t * (end_pos[2] - start_pos[2])
        
        # Create query box at this position
        query_bbox = {
            'min_x': x, 'max_x': x + 2,
            'min_y': y, 'max_y': y + 2,
            'min_z': z, 'max_z': z + 2,
            'center': (x + 1, y + 1, z + 1)
        }
        
        # Find collisions
        collisions = [i for i, bbox in enumerate(bboxes) if boxes_overlap(query_bbox, bbox)]
        
        frames.append({
            'position': (x, y, z),
            'bbox': query_bbox,
            'collisions': collisions
        })
    
    return frames

# Create frames
frames = create_path_animation_frames(cells, bboxes, num_steps=30)

print(f"Created {len(frames)} animation frames")
print(f"\nSample frames:")
for i in [0, 10, 20, 30]:
    if i < len(frames):
        f = frames[i]
        print(f"  Frame {i}: pos=({f['position'][0]:.1f}, {f['position'][1]:.1f}, {f['position'][2]:.1f}), collisions={len(f['collisions'])}")

In [ ]:
# Visualize final frame
final_frame = frames[-1]
final_query_cell = tf.Cell.Box(
    final_frame['position'][0], 
    final_frame['position'][1], 
    final_frame['position'][2], 
    2, 2, 2
)

fig_final = visualize_collision_query(
    cells, bboxes, 
    final_query_cell, 
    final_frame['bbox'], 
    final_frame['collisions']
)
fig_final.update_layout(title=f"Final Position: {len(final_frame['collisions'])} Potential Collisions")
fig_final.show()

## Note on topologic_fast BVH Support

The original topologicpy notebook uses dedicated BVH classes:

**Features not available in topologic_fast:**
- `BVH.ByTopologies()` - Create BVH from a list of topologies
- `BVH.Graph()` - Get the graph structure of the BVH
- `BVH.QueryByTopologies()` - Query BVH with topologies
- `BVH.Clashes()` - Find clashing/colliding objects

However, the core concepts demonstrated in this notebook are:
1. **Bounding Box Computation** - Using vertex coordinates
2. **Hierarchical Spatial Organization** - Building a tree structure
3. **Collision Detection** - Box overlap tests
4. **Spatial Queries** - Finding objects in a region

These can be implemented using topologic_fast's geometry primitives and graph capabilities.

## Summary

This notebook demonstrated Bounding Volume Hierarchy concepts:

1. **Creating Objects** - Grid of 3D cells using `tf.Cell.Box()`
2. **Computing Bounding Boxes** - Axis-aligned bounding boxes (AABB)
3. **Building BVH Tree** - Recursive spatial median split
4. **Visualizing Hierarchy** - Color-coded depth levels
5. **Collision Detection** - Query with bounding box overlap
6. **Path Animation** - Moving object collision tracking

### Key Concepts:
- **AABB (Axis-Aligned Bounding Box)**: Simple box aligned with coordinate axes
- **BVH Tree**: Binary tree organizing objects spatially
- **Spatial Queries**: O(log n) average case for finding nearby objects
- **Collision Detection**: First test bounding boxes, then detailed geometry

### Applications:
- Video games (collision detection)
- Ray tracing (acceleration structure)
- CAD (interference checking)
- Robotics (motion planning)